# Asistente de Bash con Claude

Escribe tu pregunta en `preguntar("...")` y Claude te responde con el comando y una explicación breve.
La conversación se recuerda, así que puedes hacer preguntas de seguimiento. Usa `nueva_conversacion()` para empezar de cero.

In [ ]:
%pip install -q anthropic

In [ ]:
import anthropic
from IPython.display import Markdown, display

client = anthropic.Anthropic()  # usa la variable de entorno ANTHROPIC_API_KEY
MODELO = "claude-opus-5-5"

SISTEMA = """Eres un experto en Bash y en la línea de comandos de Linux.
Responde en español. Para cada pregunta:
1. Da primero el comando (o script) en un bloque de código ```bash.
2. Explica brevemente qué hace cada parte y las opciones importantes.
3. Si el comando puede borrar o sobrescribir datos, adviértelo claramente.
Sé conciso."""

historial = []

def nueva_conversacion():
    historial.clear()
    print("Conversación reiniciada.")

def preguntar(pregunta, esfuerzo="low"):
    historial.append({"role": "user", "content": pregunta})
    with client.beta.messages.stream(
        model=MODELO,
        max_tokens=16000,
        system=SISTEMA,
        messages=historial,
        output_config={"effort": esfuerzo},  # low | medium | high
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
    ) as stream:
        for texto in stream.text_stream:
            print(texto, end="", flush=True)
        respuesta = stream.get_final_message()
    print()
    historial.append({"role": "assistant", "content": respuesta.content})
    if respuesta.stop_reason == "refusal":
        print("(Claude no respondió a esta pregunta.)")


## Haz tus preguntas
Cambia el texto y ejecuta la celda. Para preguntas difíciles usa `esfuerzo="high"`.

In [ ]:
preguntar("como entrar a una scrreen?")

In [ ]:
preguntar("¿Y cómo los comprimo con gzip en paralelo?")

In [ ]:
# nueva_conversacion()